In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.metrics import (
    mean_squared_error, r2_score,
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score,
    precision_score, recall_score, f1_score
)



print("=" * 70)
print("STEP 1: LOAD THE CLEANED DATA AND DEFINE OUR TARGETS")
print("=" * 70)

df = pd.read_csv('cleaned_data.csv')
print("Loaded shape:", df.shape)


# We are building TWO separate targets ("labels") from this same data:

#   y_reg = Cholesterol       -> a CONTINUOUS number (regression target)
#   y_clf = HeartDisease      -> a YES/NO column (classification target),
#                                 converted to 1 (Yes) / 0 (No)

y_reg = df['Cholesterol']
y_clf = (df['HeartDisease'] == 'Yes').astype(int)

X_raw = df.drop(columns=['PatientID', 'Cholesterol', 'HeartDisease', 'DiagnosisYear'])

print("\nRegression target (y_reg): Cholesterol (continuous)")
print("Classification target (y_clf): HeartDisease, encoded as 1=Yes, 0=No")
print("\nFeature columns before encoding:", X_raw.columns.tolist())


print("\n" + "=" * 70)
print("STEP 2: ENCODE CATEGORICAL COLUMNS")
print("=" * 70)

# Our categorical columns are: Sex, Smoker, BloodType, FamilyHistory, Diabetes

# None of these have a natural ORDER. For example, BloodType values like
# "A+", "O-", "AB+" aren't "more" or "less" than each other -- there's no
# sense in it

# Because none of them are ordinal, we use ONE-HOT ENCODING for all of
# them via pd.get_dummies(). One-hot encoding creates a separate 0/1
# column for each category instead of assigning arbitrary numbers.


categorical_cols = ['Sex', 'Smoker', 'BloodType', 'FamilyHistory', 'Diabetes']

X = pd.get_dummies(X_raw, columns=categorical_cols, drop_first=True)
print("Feature columns AFTER one-hot encoding:")
print(X.columns.tolist())
print("\nFinal X shape:", X.shape)


print("\n" + "=" * 70)
print("STEP 3: LEAK-FREE TRAIN-TEST SPLIT AND SCALING")
print("=" * 70)

X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)
print(f"Training rows: {X_train.shape[0]}, Test rows: {X_test.shape[0]}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



print("\n" + "=" * 70)
print("STEP 4: REGRESSION MODEL - LINEAR REGRESSION")
print("=" * 70)

lin_reg = LinearRegression()
lin_reg.fit(X_train_scaled, y_reg_train)
y_pred_reg = lin_reg.predict(X_test_scaled)

mse_lin = mean_squared_error(y_reg_test, y_pred_reg)
r2_lin = r2_score(y_reg_test, y_pred_reg)
print(f"\nLinear Regression - MSE: {mse_lin:.3f}")
print(f"Linear Regression - R^2: {r2_lin:.4f}")

coefficients = pd.Series(lin_reg.coef_, index=X.columns).sort_values(
    key=lambda s: s.abs(), ascending=False
)
print("\n--- All coefficients (sorted by strength) ---")
print(coefficients)

top_3_features = coefficients.head(3)
print("\n--- Top 3 features by absolute coefficient size ---")
print(top_3_features)


ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_reg_train)
y_pred_ridge = ridge.predict(X_test_scaled)

mse_ridge = mean_squared_error(y_reg_test, y_pred_ridge)
r2_ridge = r2_score(y_reg_test, y_pred_ridge)

comparison_table = pd.DataFrame({
    'Model': ['Linear Regression (OLS)', 'Ridge (alpha=1.0)'],
    'MSE': [mse_lin, mse_ridge],
    'R2': [r2_lin, r2_ridge]
})
print("\n--- Linear Regression vs Ridge ---")
print(comparison_table.to_string(index=False))


print("\n" + "=" * 70)
print("STEP 5: CLASSIFICATION MODEL - LOGISTIC REGRESSION ")
print("=" * 70)

class_counts_before = y_clf_train.value_counts()
class_percent_before = y_clf_train.value_counts(normalize=True) * 100
print("\n--- y_clf_train class counts (0 = No, 1 = Yes) ---")
print(class_counts_before)
print("\n--- As percentages ---")
print(class_percent_before.round(2))

minority_percent = class_percent_before.min()
print(f"\nSmallest class makes up {minority_percent:.2f}% of the training data.")

if minority_percent < 35:
    print("This is BELOW the 35% threshold -> imbalance correction is needed.")
else:
    print("This is ABOVE the 35% threshold -> the classes are close enough to "
          "balanced that no correction (SMOTE / class_weight) is required. "
          "We proceed with a standard LogisticRegression.")

log_reg = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
log_reg.fit(X_train_scaled, y_clf_train)

y_pred_clf = log_reg.predict(X_test_scaled)                    # 0/1 class predictions
y_proba_clf = log_reg.predict_proba(X_test_scaled)[:, 1]       # probability of class 1

# Confusion matrix:

cm = confusion_matrix(y_clf_test, y_pred_clf)
print("\n--- Confusion Matrix ---")
print(cm)
tn, fp, fn, tp = cm.ravel()
print(f"True Negatives: {tn}, False Positives: {fp}, "
      f"False Negatives: {fn}, True Positives: {tp}")

print("\n--- Classification Report (precision, recall, F1, accuracy) ---")
print(classification_report(y_clf_test, y_pred_clf, target_names=['No', 'Yes']))

#roc curve:
fpr, tpr, roc_thresholds = roc_curve(y_clf_test, y_proba_clf)
auc_baseline = roc_auc_score(y_clf_test, y_proba_clf)
print(f"\nAUC (C=1.0 model): {auc_baseline:.4f}")

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='#2980b9', linewidth=2, label=f'Logistic Regression (AUC = {auc_baseline:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random guess (AUC = 0.5)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Heart Disease Classification')
plt.legend(loc='lower right')
plt.annotate(f'AUC = {auc_baseline:.3f}', xy=(0.6, 0.2), fontsize=12,
             bbox=dict(boxstyle='round', facecolor='white', edgecolor='#2980b9'))
plt.tight_layout()
plt.savefig('plots/7_roc_curve.png', dpi=120)
plt.close()
print("Saved: plots/7_roc_curve.png")


print("\n" + "=" * 70)
print("STEP 5b: DECISION-THRESHOLD SENSITIVITY")
print("=" * 70)

threshold_rows = []
for threshold in [0.30, 0.40, 0.50, 0.60, 0.70]:
    preds_at_threshold = (y_proba_clf >= threshold).astype(int)
    precision = precision_score(y_clf_test, preds_at_threshold, zero_division=0)
    recall = recall_score(y_clf_test, preds_at_threshold, zero_division=0)
    f1 = f1_score(y_clf_test, preds_at_threshold, zero_division=0)
    threshold_rows.append({'Threshold': threshold, 'Precision': precision,
                            'Recall': recall, 'F1': f1})

threshold_table = pd.DataFrame(threshold_rows)
print("\n--- Precision / Recall / F1 at different thresholds ---")
print(threshold_table.round(4).to_string(index=False))

best_threshold_row = threshold_table.loc[threshold_table['F1'].idxmax()]
print(f"\nThreshold that maximizes F1: {best_threshold_row['Threshold']} "
      f"(F1 = {best_threshold_row['F1']:.4f})")


print("\n" + "=" * 70)
print("STEP 6: REGULARIZATION EXPERIMENT (C=1.0 vs C=0.01)")
print("=" * 70)

log_reg_strong = LogisticRegression(max_iter=1000, C=0.01, random_state=42)
log_reg_strong.fit(X_train_scaled, y_clf_train)

y_pred_strong = log_reg_strong.predict(X_test_scaled)
y_proba_strong = log_reg_strong.predict_proba(X_test_scaled)[:, 1]

precision_base = precision_score(y_clf_test, y_pred_clf, zero_division=0)
recall_base = recall_score(y_clf_test, y_pred_clf, zero_division=0)
precision_strong = precision_score(y_clf_test, y_pred_strong, zero_division=0)
recall_strong = recall_score(y_clf_test, y_pred_strong, zero_division=0)
auc_strong = roc_auc_score(y_clf_test, y_proba_strong)

regularization_table = pd.DataFrame({
    'Model': ['C=1.0 (baseline)', 'C=0.01 (strong regularization)'],
    'Precision': [precision_base, precision_strong],
    'Recall': [recall_base, recall_strong],
    'AUC': [auc_baseline, auc_strong]
})
print("\n--- C=1.0 vs C=0.01 ---")
print(regularization_table.round(4).to_string(index=False))


print("\n" + "=" * 70)
print("STEP 7: BOOTSTRAP CONFIDENCE INTERVAL FOR THE AUC DIFFERENCE")
print("=" * 70)

np.random.seed(42)
n_bootstrap = 500
y_clf_test_array = y_clf_test.values
auc_differences = []

for i in range(n_bootstrap):
    # Randomly pick row indices, WITH replacement, same size as the test set
    sample_indices = np.random.choice(len(y_clf_test_array), size=len(y_clf_test_array), replace=True)

    y_sample = y_clf_test_array[sample_indices]
    proba_baseline_sample = y_proba_clf[sample_indices]
    proba_strong_sample = y_proba_strong[sample_indices]

    # A bootstrap sample could (rarely) end up with only one class present,
    # which makes AUC undefined -- skip those rare cases
    if len(np.unique(y_sample)) < 2:
        continue

    auc_baseline_sample = roc_auc_score(y_sample, proba_baseline_sample)
    auc_strong_sample = roc_auc_score(y_sample, proba_strong_sample)
    auc_differences.append(auc_baseline_sample - auc_strong_sample)

auc_differences = np.array(auc_differences)
mean_auc_difference = auc_differences.mean()
ci_lower = np.percentile(auc_differences, 2.5)
ci_upper = np.percentile(auc_differences, 97.5)

print(f"\nBootstrap iterations used: {len(auc_differences)} (out of {n_bootstrap} attempted)")
print(f"Mean AUC difference (C=1.0 minus C=0.01): {mean_auc_difference:.5f}")
print(f"95% Confidence Interval: [{ci_lower:.5f}, {ci_upper:.5f}]")

if ci_lower > 0 or ci_upper < 0:
    print("The interval EXCLUDES zero -> the AUC difference looks reliable/consistent.")
else:
    print("The interval INCLUDES zero -> we cannot confidently say one model's "
          "AUC is truly higher than the other's; the observed difference may just be noise.")





STEP 1: LOAD THE CLEANED DATA AND DEFINE OUR TARGETS
Loaded shape: (650, 16)

Regression target (y_reg): Cholesterol (continuous)
Classification target (y_clf): HeartDisease, encoded as 1=Yes, 0=No

Feature columns before encoding: ['Age', 'Sex', 'BMI', 'Smoker', 'BloodType', 'SystolicBP', 'DiastolicBP', 'GlucoseMgDl', 'RestingHR', 'ExerciseHrsPerWeek', 'FamilyHistory', 'Diabetes']

STEP 2: ENCODE CATEGORICAL COLUMNS
Feature columns AFTER one-hot encoding:
['Age', 'BMI', 'SystolicBP', 'DiastolicBP', 'GlucoseMgDl', 'RestingHR', 'ExerciseHrsPerWeek', 'Sex_Male', 'Smoker_Yes', 'BloodType_A-', 'BloodType_AB+', 'BloodType_AB-', 'BloodType_B+', 'BloodType_B-', 'BloodType_O+', 'BloodType_O-', 'FamilyHistory_Yes', 'Diabetes_Yes']

Final X shape: (650, 18)

STEP 3: LEAK-FREE TRAIN-TEST SPLIT AND SCALING
Training rows: 520, Test rows: 130

STEP 4: REGRESSION MODEL - LINEAR REGRESSION

Linear Regression - MSE: 1144.868
Linear Regression - R^2: -0.0637

--- All coefficients (sorted by strength) --